# 02 · DAG Anatomy & Scheduling

> **Module 1 · Composer Fundamentals & Model · Lecture 1.3**

In this notebook you will:

- The anatomy of a `DAG` object — every parameter and what it actually controls
- Distinguish the **logical date**, the **data interval** and the **execution date**
- Internalise the half-open interval convention `[start, end)` and the off-by-one it prevents
- Understand `schedule`, `catchup`, `start_date` and `max_active_runs` and their interactions
- Build a schedule simulator you can run, so `catchup` stops being a mystery

## 1. The DAG object

Everything about a DAG's *behaviour* comes from its constructor. The course's
`01_hello_world.py` is the anchor:

```python
with DAG(
    dag_id="hello_world",
    description="The smallest useful DAG — verify your Composer environment.",
    schedule="@daily",
    start_date=datetime(2026, 1, 1, tzinfo=timezone.utc),
    catchup=False,
    max_active_runs=1,
    default_args=DEFAULT_ARGS,
    tags=["tutorial", "composer"],
    params={"audience": "data-engineers"},
) as dag:
    ...
```

| Parameter | Meaning | Default | Notes |
|---|---|---|---|
| `dag_id` | unique identifier | required | must be unique across the environment |
| `schedule` | when to create runs | `None` | cron / timedelta / dataset / `None` |
| `start_date` | earliest logical date to consider | required | timezone-aware, strongly recommended |
| `catchup` | create runs for missed intervals | `False` | the dangerous one |
| `max_active_runs` | concurrent runs of this DAG | `16` | `1` for fixed-destination pipelines |
| `max_active_tasks` | concurrent tasks across all runs | `16` | rarely changed |
| `default_args` | default kwargs for every task | `{}` | owner, retries, retry_delay |
| `params` | DAG-run parameters with defaults | `{}` | overridable per run |
| `sla` | time allowed after the data interval ends | `None` | measured from interval **end** |
| `sla_miss_callback` | called on SLA miss | `None` | receives the task context |
| `tags` | labels for the UI | `[]` | filterable in the UI |
| `dagrun_timeout` | max seconds a run may take | `None` | kills the whole run |
| `schedule_interval` | **old name for `schedule`** | | deprecated since 2.4 |

The mini-runner's `DAG` implements the same shape, minus the Airflow-only
parts. Let's look at it.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

import inspect

signature = inspect.signature(local_runner.DAG.__init__)
print("DAG.__init__ parameters:\n")
for name, param in signature.parameters.items():
    if name == "self":
        continue
    default = (
        param.default
        if param.default is not inspect.Parameter.empty
        else "(required)"
    )
    print(f"  {name:<18} default={default}")

## 2. `start_date` — a floor, not a trigger

`start_date` is the **earliest logical date Airflow will consider**. It is
*not* "when the DAG was created" and *not* "when the first run happens".

Three rules:

1. **It should be timezone-aware.** Use
   `datetime(2026, 1, 1, tzinfo=timezone.utc)`. A naive datetime is
   interpreted in the Airflow server's local timezone, which differs between
   your laptop, a self-hosted server, and Composer. The course always uses UTC.
2. **It is a floor, not a trigger.** Setting `start_date` to `2026-01-01` does
   not create 20 runs for January. It says: *if* runs are to be created for
   dates on or after `2026-01-01`, this DAG is eligible.
3. **Changing it retroactively can create history.** Lower `start_date` with
   `catchup=True` and the scheduler will want runs for the newly-eligible
   interval.

## 3. `schedule` — when a run is *created*

`schedule` decides *when a run is created*. It does not decide when a task
runs, and it does not decide what data the run processes — that is the logical
date's job.

| Form | Example | Meaning |
|---|---|---|
| **Preset** | `"@daily"` | once a day, at midnight UTC |
| | `"@hourly"` | once an hour, on the hour |
| | `"@weekly"` | Mondays at midnight UTC |
| | `"@monthly"` | 1st of the month at midnight UTC |
| | `"@reboot"` | once, when the scheduler starts |
| **Cron** | `"0 6 * * *"` | 06:00 every day (UTC) |
| | `"30 2 * * 0"` | 02:30 every Sunday |
| **Timedelta** | `timedelta(hours=6)` | every six hours from `start_date` |
| **Dataset** | `[orders_dataset]` | when the dataset is updated (Airflow 2.4+) |
| **None** | `None` | never scheduled; manual trigger only |

Two Airflow-version notes that matter:

- **`schedule_interval` → `schedule`.** Renamed in Airflow 2.4. The old name
  still works but is deprecated. **Use `schedule`.**
- **Cron is evaluated in UTC** by default in Airflow 2.x. A `0 6 * * *` cron
  fires at 06:00 UTC, not 06:00 local. Composer does not change this.

## 4. The three dates

This is the concept most worth getting exactly right, because every
downstream bug traces back to it.

| Name | What it is |
|---|---|
| **Logical date** | the *label* of the run — the start of the interval it processes |
| **Data interval** | the half-open range `[start, end)` the run is responsible for |
| **Execution date** | when the scheduler actually created the run |

For a `@daily` DAG with `logical_date = 2026-01-15`:

- the **data interval** is `[2026-01-15T00:00:00Z, 2026-01-16T00:00:00Z)`;
- the **logical date** is `2026-01-15T00:00:00Z`;
- the **execution date** is whenever the scheduler got around to it — say
  `2026-01-16T03:14:22Z` if the previous run overran.

The name "logical date" is deliberately confusing; Airflow renamed it from
"execution date" in 2.2 precisely because the old name was wrong. The logical
date tells you **which data to process**. The execution date tells you
**when the work happened**.

### The rule that follows

> **Partition and filter by the logical date. Never by "now".**

If a task queries `WHERE event_timestamp >= NOW() - INTERVAL 1 DAY`, then a
backfill run in March processes March's data using March's `NOW()` — which is
wrong for a backfill of January, and silently produces empty results. The
logical date pins the window.

The course models this exactly. `pipeline_logic.logical_date_window` turns a
logical date into a `[start, end)` window.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from pipeline_logic import (
    transform_orders,
    summarise_by_region,
    render_summary_csv,
    render_rejects_csv,
    reject_counts,
    logical_date_window,
    partition_for_date,
)

print("logical_date_window('2026-01-15')")
print("  ->", logical_date_window("2026-01-15"))
print()
print("The window is half-open: start is INCLUDED, end is EXCLUDED.\n")

for days in (1, 7, 30):
    start, end = logical_date_window("2026-01-15", data_interval_days=days)
    print(f"  interval_days={days:<3} -> [{start}, {end})")

print()
print("An inclusive variant exists for systems whose filters are inclusive:")
print(
    "  include_end=True ->",
    logical_date_window("2026-01-15", include_end=True),
)
print(
    "  include_end=True ->",
    logical_date_window("2026-01-15", data_interval_days=7, include_end=True),
)

### Why half-open beats closed intervals

Suppose the filter were **inclusive** on both ends: `BETWEEN start AND end`.
With a daily schedule, `2026-01-15` would appear in the `2026-01-15` run
*and* the `2026-01-16` run — processed twice.

With half-open `[start, end)`, adjacent intervals **partition** the timeline
with no gaps and no overlap:

```text
[2026-01-15, 2026-01-16)   [2026-01-16, 2026-01-17)   [2026-01-17, 2026-01-18)
        ↑                                    ↑
   01-15T00:00:00Z                   01-15T23:59:59Z belongs to
   belongs to interval 1             interval 1, not interval 2
```

Let's prove the partitioning property directly — this is the kind of check
that belongs in your test suite.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from pipeline_logic import (
    transform_orders,
    summarise_by_region,
    render_summary_csv,
    render_rejects_csv,
    reject_counts,
    logical_date_window,
    partition_for_date,
)

from datetime import date, timedelta

# Build 7 consecutive daily windows and check they tile perfectly.
start = date(2026, 1, 15)
windows = [
    logical_date_window((start + timedelta(days=i)).isoformat())
    for i in range(7)
]

print("7 consecutive daily windows:")
for w_start, w_end in windows:
    print(f"  [{w_start}, {w_end})")

# Property 1: each window's end is the next window's start (no gaps).
contiguous = all(
    windows[i][1] == windows[i + 1][0] for i in range(len(windows) - 1)
)
# Property 2: each window is exactly one day wide.
one_day = all(
    (date.fromisoformat(e) - date.fromisoformat(s)).days == 1
    for s, e in windows
)
# Property 3: no instant is in two windows (half-open => disjoint).
disjoint = all(
    date.fromisoformat(windows[i][1])
    <= date.fromisoformat(windows[i + 1][0])
    for i in range(len(windows) - 1)
)

print()
print(f"contiguous (no gaps)      : {contiguous}")
print(f"exactly one day wide      : {one_day}")
print(f"disjoint (no double-count): {disjoint}")
print()
print("PASS: the intervals tile the timeline exactly once each.")

## 5. `catchup` — and why it is dangerous

`catchup` decides whether the scheduler creates runs for intervals that
already passed while the DAG was not running (or was broken).

- `catchup=False` (the default, and the right default) — the scheduler creates
  only the *current* interval. Missed intervals are simply not run. You
  backfill deliberately, on purpose.
- `catchup=True` — the scheduler creates a run for **every** elapsed interval
  since `start_date`. Deploy a DAG with `start_date=2026-01-01` and
  `catchup=True` on `2026-03-01` and you get ~60 runs, immediately, all
  writing to the same destination table.

That last scenario is the classic incident: a backfill of 60 daily runs, each
doing a `WRITE_TRUNCATE`, wiping the table 60 times. Or each doing an
`WRITE_APPEND`, duplicating every row 60 times.

### A schedule simulator

Rather than describe catchup, let's *build* the thing. The simulator below
models what the Airflow scheduler does: given a `start_date`, a `schedule`, a
`now`, and a `catchup` flag, produce the list of logical dates it will
create.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from pipeline_logic import (
    transform_orders,
    summarise_by_region,
    render_summary_csv,
    render_rejects_csv,
    reject_counts,
    logical_date_window,
    partition_for_date,
)

import datetime as dt


def parse_schedule(schedule: str) -> dt.timedelta:
    """Convert a schedule string into an interval length."""
    presets = {
        "@hourly": dt.timedelta(hours=1),
        "@daily": dt.timedelta(days=1),
        "@weekly": dt.timedelta(days=7),
        "@monthly": dt.timedelta(days=30),
    }
    if schedule in presets:
        return presets[schedule]
    if schedule.startswith("@"):
        raise ValueError(f"unsupported preset: {schedule}")
    # Treat a 5-field cron as daily for the purposes of this simulator.
    parts = schedule.split()
    if len(parts) == 5:
        if parts[0] == "*" and parts[1] == "*" and parts[2] == "*" and parts[3] == "*":
            return dt.timedelta(days=1)
        raise ValueError(f"unsupported cron: {schedule} (hourly crons not modelled)")
    raise ValueError(f"unsupported schedule: {schedule}")


def simulate_schedule(
    start_date: str,
    schedule: str,
    now: str,
    catchup: bool,
) -> list[str]:
    """Simulate which logical dates the Airflow scheduler would create."""
    interval = parse_schedule(schedule)
    start = dt.datetime.strptime(start_date, "%Y-%m-%d")
    now_dt = dt.datetime.strptime(now, "%Y-%m-%d %H:%M")

    logical_dates: list[str] = []
    current = start
    while current < now_dt:
        logical_dates.append(current.date().isoformat())
        current += interval

    if not catchup:
        # Without catchup, only the most recent interval is created.
        logical_dates = logical_dates[-1:] if logical_dates else []

    return logical_dates


print("Simulator built.\n")
print("--- catchup=False, DAG deployed late ---")
print(simulate_schedule("2026-01-01", "@daily", "2026-03-01 12:00", catchup=False))
print()
print("--- catchup=True, same situation ---")
print(simulate_schedule("2026-01-01", "@daily", "2026-03-01 12:00", catchup=True))

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from pipeline_logic import (
    transform_orders,
    summarise_by_region,
    render_summary_csv,
    render_rejects_csv,
    reject_counts,
    logical_date_window,
    partition_for_date,
)

print("The difference in one picture:\n")
print(f"{'logical date':<14} {'catchup=False':<16} {'catchup=True':<16}")
print("-" * 48)

no_catchup = set(
    simulate_schedule("2026-01-01", "@daily", "2026-03-01 12:00", False)
)
all_dates = simulate_schedule(
    "2026-01-01", "@daily", "2026-03-01 12:00", True
)

for date_str in all_dates:
    mark = "  (skipped)" if date_str not in no_catchup else ""
    created = "created" if date_str in no_catchup else "not created"
    print(f"{date_str:<14} {created:<16} {'created':<16}{mark}")

print()
print(f"catchup=False creates {len(no_catchup)} run(s).")
print(f"catchup=True  creates {len(all_dates)} runs.")

That is the whole danger, quantified: **1 run versus 60**, from the
same DAG, differing only in one boolean.

Note the subtlety in the simulator: with `catchup=False`, only the **most
recent** interval is created — not "all intervals from now on", but the
single latest one, and then one per subsequent tick.

### The safe way to backfill

Never rely on `catchup=True` for a production backfill. Instead:

1. Trigger runs **manually**, one logical date at a time, with an explicit
   `conf`/`params` carrying the date range.
2. Set `max_active_runs=1` so backfill runs cannot overlap each other or the
   live schedule.
3. Use `WRITE_TRUNCATE` scoped to the partition, or a deterministic
   deduplicating write — see notebook 06.
4. Run the backfill on a copy first, and diff the output.

Composer gives you this in the UI: **DAG → Trigger DAG w/ config**, plus the
**"Trigger DAG" / "Create a DAG run"** forms. And the Airflow CLI
`airflow dags backfill` exists for larger jobs — but in Composer, the API
`POST /api/v1/dags/{dag_id}/dagRuns` is the reliable path.

## 6. `max_active_runs` and `max_active_tasks`

These two control concurrency, and the distinction is:

- **`max_active_runs`** — how many *runs* of this DAG may be in flight at
  once. Default `16`.
- **`max_active_tasks`** — how many *tasks* (across all runs) may execute
  concurrently. Default `16`.

For a pipeline that writes to a fixed destination, `max_active_runs=1` is
usually correct: two concurrent runs writing to the same table is a race, and
it is a race that only manifests under load, which is the worst time to
discover it.

The course's `01_hello_world.py` uses `max_active_runs=1` for exactly this
reason.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from pipeline_logic import (
    transform_orders,
    summarise_by_region,
    render_summary_csv,
    render_rejects_csv,
    reject_counts,
    logical_date_window,
    partition_for_date,
)

dag = local_runner.DAGS["etl_to_gcs"]()

print(f"dag_id          : {dag.dag_id}")
print(f"schedule        : {dag.schedule}")
print(f"start_date      : {dag.start_date}")
print(f"catchup         : {dag.catchup}")
print(f"max_active_runs : {dag.max_active_runs}")
print(f"description     : {dag.description}")
print(f"tasks           : {len(dag.tasks)}")

print()
print("Downstream of 'transform_orders':")
for task_id in dag.downstream("transform_orders"):
    print(f"  -> {task_id}")

## 7. Dependencies, ordering and the DAG invariant

Airflow uses `>>` to set dependencies:

```python
read_orders >> transform_orders >> aggregate >> write_summary
```

The mini-runner expresses the same thing with explicit `dependencies=[...]`.
Both mean the same thing: **the right-hand task runs only after every
left-hand task has succeeded.**

Two invariants are worth checking explicitly, because Airflow validates them
at parse time and a violation means your DAG never appears in the UI:

1. **No cycles.** Acyclic ⇒ a topological order exists ⇒ "run everything
   ready" is well-defined.
2. **No unknown dependencies.** A typo in a task_id is a parse error, not a
   runtime surprise.

The mini-runner enforces both in its constructor. Let's watch it reject bad
DAGs.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from pipeline_logic import (
    transform_orders,
    summarise_by_region,
    render_summary_csv,
    render_rejects_csv,
    reject_counts,
    logical_date_window,
    partition_for_date,
)

from local_runner import DAG, Task

# 1. A typo in a dependency name.
try:
    DAG(
        "typo_dag",
        tasks=[
            Task("read_orders", lambda run, context: None, dependencies=[]),
            Task(
                "transform",
                lambda run, context: None,
                dependencies=["transform_orders"],
            ),
        ],
    )
except ValueError as exc:
    print("REJECTED unknown dependency:")
    print(f"  {exc}")
print()

# 2. A cycle: a -> b -> a
try:
    DAG(
        "cyclic_dag",
        tasks=[
            Task("a", lambda run, context: None, dependencies=["b"]),
            Task("b", lambda run, context: None, dependencies=["a"]),
        ],
    )
except ValueError as exc:
    print("REJECTED cycle:")
    print(f"  {exc}")

Airflow raises the equivalent errors at parse time — which is a
feature. A DAG that fails to parse never gets scheduled, so it cannot
half-run.

## 8. Execution order is topological, not textual

A subtle and important property: **the order tasks are written in the file
does not determine the order they run in.** Airflow schedules by dependency,
not by position.

The runner implements the same rule with `_dependency_order`, which recomputes
a topological ordering on each pass — recomputed, because a branch skip can
unblock tasks that were previously blocked.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from pipeline_logic import (
    transform_orders,
    summarise_by_region,
    render_summary_csv,
    render_rejects_csv,
    reject_counts,
    logical_date_window,
    partition_for_date,
)

from local_runner import DAG, Task, _dependency_order

# A DAG written in the WRONG textual order: write_summary is declared first.
reversed_dag = DAG(
    "reversed",
    tasks=[
        Task("write_summary", lambda run, context: None, dependencies=["aggregate"]),
        Task("aggregate", lambda run, context: None, dependencies=["transform_orders"]),
        Task("transform_orders", lambda run, context: None, dependencies=["read_orders"]),
        Task("read_orders", lambda run, context: None, dependencies=[]),
    ],
)

order = _dependency_order(reversed_dag, set(reversed_dag.tasks))
print("Declaration order :", list(reversed_dag.tasks))
print("Execution order   :", order)
print()
print("Dependencies always come first, regardless of declaration order.")
print("Ties are broken by declaration order, so runs are deterministic.")

## 9. Building a DAG in the mini-runner

Now let's build one from scratch and run it at several logical dates — the
same DAG, four dates, four data windows. That is the practical payoff of
everything above.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from pipeline_logic import (
    transform_orders,
    summarise_by_region,
    render_summary_csv,
    render_rejects_csv,
    reject_counts,
    logical_date_window,
    partition_for_date,
)

from local_runner import DAG, Task


def read_orders(run, context):
    """Read the raw CSV."""
    path = COURSE_ROOT / "sample_data" / "orders.csv"
    return path.read_text(encoding="utf-8").splitlines()


def transform(run, context):
    """Validate and transform."""
    raw = run.pull("read_orders") or []
    accepted, rejected = transform_orders(raw)
    run.xcom.push("transform", "accepted", accepted)
    run.xcom.push("transform", "rejected", rejected)
    return {"accepted": len(accepted), "rejected": len(rejected)}


def aggregate(run, context):
    """Summarise by region."""
    accepted = run.pull("transform", "accepted") or []
    summary = summarise_by_region(accepted)
    run.xcom.push("aggregate", "summary", summary)
    return summary


def write_summary(run, context):
    """Write the summary, tagged with the run's logical date."""
    summary = run.pull("aggregate", "summary") or {}
    window = logical_date_window(run.logical_date)
    out = COURSE_ROOT / "out" / "notebook02" / run.logical_date / "summary.csv"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(render_summary_csv(summary), encoding="utf-8")
    return {
        "path": str(out),
        "window": window,
        "rows": render_summary_csv(summary),
    }


demo = DAG(
    "scheduling_demo",
    tasks=[
        Task("read_orders", read_orders, dependencies=[]),
        Task("transform", transform, dependencies=["read_orders"]),
        Task("aggregate", aggregate, dependencies=["transform"]),
        Task("write_summary", write_summary, dependencies=["aggregate"]),
    ],
    schedule="@daily",
    start_date="2026-01-15",
    catchup=False,
    max_active_runs=1,
    description="Demonstrates logical date, data interval and execution order.",
)

print(demo.describe())

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from pipeline_logic import (
    transform_orders,
    summarise_by_region,
    render_summary_csv,
    render_rejects_csv,
    reject_counts,
    logical_date_window,
    partition_for_date,
)

print("The same DAG, four logical dates:\n")
print(f"{'logical date':<14} {'data interval':<34} {'accepted':>9}")
print("-" * 60)

for logical_date in ("2026-01-13", "2026-01-14", "2026-01-15", "2026-01-16"):
    run = local_runner.run_dag(demo, logical_date=logical_date)
    window = logical_date_window(logical_date)
    accepted = run.pull("transform", "accepted") or []
    states = set(run.state.values())
    status = "ok" if states == {"success"} else states
    print(
        f"{logical_date:<14} [{window[0]}, {window[1]}){'':<9} "
        f"{len(accepted):>9}  {status}"
    )

print()
print("Note: the source file is a single day's snapshot, so the accepted")
print("count is identical for every date. In a real DAG the source would")
print("be filtered to the run's data interval - that filter is what makes")
print("the logical date meaningful.")

## 10. Cross-DAG dependencies and `schedule=None`

A DAG with `schedule=None` is **never scheduled** — it runs only when
triggered. This is how you build a pipeline that is triggered by an *event*
rather than by time:

- the parent DAG finishes → `TriggerDagRunOperator` fires the child;
- an upstream system lands data → a sensor waits, then triggers;
- a human clicks "Run" in the UI.

`04_branching_and_xcom.py` in the course uses exactly this pattern.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from pipeline_logic import (
    transform_orders,
    summarise_by_region,
    render_summary_csv,
    render_rejects_csv,
    reject_counts,
    logical_date_window,
    partition_for_date,
)

# A child DAG with no schedule - manual/event triggered only.
child = DAG(
    "child_pipeline",
    tasks=[
        Task("read_orders", read_orders, dependencies=[]),
        Task("transform", transform, dependencies=["read_orders"]),
    ],
    schedule=None,
    start_date="2026-01-15",
    description="Triggered by the parent, not by the clock.",
)

print(f"schedule    : {child.schedule}")
print(f"auto-runs?  : {child.schedule is not None}")
print()
print("In Composer this DAG appears in the list but has no 'Next Run' column")
print("value - it is only ever started by an explicit trigger.")

## 11. Checklist: the scheduling questions to ask of every DAG

Before you deploy a DAG, you should be able to answer all seven:

1. **What is the schedule, and in which timezone?** (UTC, unless you know why not.)
2. **What is `start_date`, and is it timezone-aware?**
3. **Is `catchup` True or False — and if True, how many runs would that
   create on deploy day?**
4. **What is `max_active_runs`, and can two runs write to the same place
   concurrently?**
5. **What is the data interval, and does every filter match it exactly?**
6. **If the run is for `2026-01-15`, does every query filter on the logical
   date rather than on `NOW()`?**
7. **What happens when a run is late — is there an SLA, and who gets paged?**

## Exercises

1. **Schedule simulator extension.** Extend `parse_schedule` to handle `"@hourly"` and a 5-field cron of the form `"0 * * * *"` (hourly). Then simulate a `@hourly` DAG with `start_date=2026-01-15`, `now=2026-01-15 06:30`, `catchup=True`, and report how many runs are created and what their logical dates are.
2. **Harder — a week of catchup.** Simulate `start_date="2026-01-01"`, `schedule="@daily"`, `now="2026-01-08 12:00"` with `catchup=True`. List every logical date. Then explain, in prose, what would happen to a BigQuery table if each of those runs did a `WRITE_TRUNCATE` load, and what would happen if each did a `WRITE_APPEND`.
3. **Harder — partition math.** Using `logical_date_window` and `partition_for_date` from `pipeline_logic`, write a function `partitions_for_backfill(start, end)` that returns the list of partition decorators for every daily interval in `[start, end)`. For `start="2026-01-13"` and `end="2026-01-16"`, what should it return, and why is the last element exclusive?

<details>
<summary><b>Solution (click to reveal)</b></summary>

```python
# Exercise 1
def parse_schedule(schedule):
    presets = {
        "@hourly": dt.timedelta(hours=1),
        "@daily": dt.timedelta(days=1),
        "@weekly": dt.timedelta(days=7),
    }
    if schedule in presets:
        return presets[schedule]
    parts = schedule.split()
    if len(parts) == 5:
        if parts[2] == "*" and parts[3] == "*" and parts[4] == "*":
            return dt.timedelta(hours=1) if parts[1] == "*" else None
        return dt.timedelta(days=1)
    raise ValueError(schedule)
```

```python
# Exercise 2
dates = simulate_schedule("2026-01-01", "@daily", "2026-01-08 12:00", catchup=True)
print(dates)   # ['2026-01-01' ... '2026-01-07'] - 7 runs
```

With `WRITE_TRUNCATE`, each run replaces the whole table with that single
day's data, so the table ends up containing only the last run's rows
(2026-01-07) — the other six days are destroyed. With `WRITE_APPEND`, each
run adds its day without removing anything, so the table ends up containing
the union of all seven days — which happens to be correct *here*, but only
because no day overlaps. The dangerous combination is `WRITE_APPEND` with a
schedule that overlaps (e.g. a 6-hourly schedule appending to a daily
partition), which quadruples every row.

```python
# Exercise 3
from pipeline_logic import partition_for_date

def partitions_for_backfill(start, end):
    partitions = []
    current = dt.date.fromisoformat(start)
    last = dt.date.fromisoformat(end)
    while current < last:                 # half-open: end is exclusive
        partitions.append(partition_for_date(current.isoformat()))
        current += dt.timedelta(days=1)
    return partitions

print(partitions_for_backfill("2026-01-13", "2026-01-16"))
# ['20260113', '20260114', '20260115']
```

The end is exclusive because the window convention is uniform: every window
in the system is `[start, end)`. Using an inclusive end here would make the
backfill overlap the next scheduled run, and the last partition would be
written twice.

</details>